<a href="https://colab.research.google.com/github/Kunj-7007/AI_ML_Workshop_LDRP/blob/Day4/GAN_GENERATOR_DISCRIMINATOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

In [4]:
#Load the dataset
data=pd.read_csv('/content/energy_architecture_data.csv')
#display first few rows
print (data.head())
print(data.shape)

   floor_area  number_of_floors  window_to_wall_ratio  wall_thickness  \
0       281.4                 2                  0.49            0.37   
1       186.7                 2                  0.30            0.45   
2       198.4                 2                  0.30            0.42   
3       154.1                 2                  0.15            0.42   
4       123.8                 2                  0.30            0.37   

   avg_temperature  avg_humidity  solar_radiation  thermal_resistance  \
0             17.1            61              415                0.10   
1             -0.4            24              116                5.18   
2             19.3            43              138                0.08   
3             -7.0            72              484                6.22   
4             23.0            67              937                3.24   

   estimated_energy_usage  daylight_factor  thermal_load  
0                 61362.6             2.03        -807.9  
1   

In [3]:
# Dimensions
input_dim = 100  # Random noise vector input for the generator
output_dim = data.shape[1]  # Number of features in the data


# Generator Model
def build_generator():
    model = tf.keras.Sequential([
        layers.Dense(128, activation="relu", input_dim=input_dim),
        layers.Dense(256, activation="relu"),
        layers.Dense(512, activation="relu"),
        layers.Dense(output_dim, activation="tanh")  # Output layer
    ])
    return model


In [5]:
# Discriminator Model
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Dense(512, activation="relu", input_shape=(output_dim,)),
        layers.Dense(256, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid")  # Binary classification
    ])
    return model


In [6]:
# Instantiating and Compiling Models
generator = build_generator()
discriminator = build_discriminator()


discriminator.compile(optimizer=tf.keras.optimizers.Adam(0.0002, 0.5), loss="binary_crossentropy", metrics=["accuracy"])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
# Building the GAN Model
discriminator.trainable = False  # Freeze the discriminator's weights when training the GAN
gan_input = layers.Input(shape=(input_dim,))
fake_data = generator(gan_input)
gan_output = discriminator(fake_data)
gan = tf.keras.Model(gan_input, gan_output)
gan.compile(optimizer=tf.keras.optimizers.Adam(0.0002, 0.5), loss="binary_crossentropy")

In [15]:
# Training the GAN Model
def train(data, batch_size, epochs=500):
    half_batch = int(batch_size / 2)


    for epoch in range(epochs):
        # --------------------- Train Discriminator ---------------------
        # Set discriminator trainable to True
        discriminator.trainable = True


        # Select a random half-batch of real samples
        idx = np.random.randint(0, data.shape[0], half_batch)
        real_data = data.iloc[idx].values


        # Generate a half-batch of fake samples
        noise = np.random.normal(0, 1, (half_batch, input_dim))
        fake_data = generator.predict(noise)


        # Train the discriminator on real and fake samples
        d_loss_real = discriminator.train_on_batch(real_data, np.ones((half_batch, 1)))
        d_loss_fake = discriminator.train_on_batch(fake_data, np.zeros((half_batch, 1)))
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)


        # --------------------- Train Generator ---------------------
        # Set discriminator trainable to False for GAN training
        discriminator.trainable = False


        # Generate a batch of noise
        noise = np.random.normal(0, 1, (batch_size, input_dim))

        # Train the generator
        g_loss = gan.train_on_batch(noise, np.ones((batch_size, 1)))


        # Print progress
        if epoch % 100 == 0:
            print(f"epoch: {epoch+1}/{epochs} [D loss: {d_loss[0]:.4f}, acc.: {100*d_loss[1]:.2f}] [G loss: {g_loss}]")
        else:
            print(f"epoch: {epoch+1}/{epochs} ...")

# Call the train function after its definition
train(data, batch_size=64, epochs=500)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
epoch: 1/500 [D loss: 0.2908, acc.: 96.14] [G loss: 1.0002425909042358]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
epoch: 2/500 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
epoch: 3/500 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
epoch: 4/500 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
epoch: 5/500 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
epoch: 6/500 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
epoch: 7/500 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
epoch: 8/500 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
epoch: 9/500 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
epoch: 10/500 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
epoch: 11/500 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
epoch: 12/500 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
epoch: 13/500 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
epoch: 14/500 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
epoch: 15/500 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
epoch: 16/500 ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
epoch: 17/500 ...
1/

In [16]:
# Function generate synthetic data
def generate_synthetic_data(generator, num_samples=100):
    noise_dim=100
    # Generate random noise
    noise = np.random.normal(0, 1, (num_samples, noise_dim))

    # Generate synthetic data using the generator
    synthetic_data = generator.predict(noise)
    return synthetic_data
